# transfer

In [ ]:
# 修改你的代码：嵌套循环跑多个训练集比例和冻结层组合

import os
import pandas as pd
import numpy as np
import torch
from torch import nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from datetime import datetime
import joblib

# 设置随机种子
random_seed = 45
np.random.seed(random_seed)
torch.manual_seed(random_seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 实验主目录
base_experiment_dir = "experiments_transfer_v3_b1_45"
os.makedirs(base_experiment_dir, exist_ok=True)

# 参数设置
hidden_dim = 64
num_layers = 3
dropout = 0.2
epochs = 500
learning_rate = 1e-4
n_past = 30
n_future = 10
batch_size = 64
timestamp_col = 'timestamp'
target_feature = 'motor6_temperature'
data_file = '../data/target_all_data_b1.csv'

# 模型定义
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, dropout=dropout if num_layers > 1 else 0, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])
        out = self.fc(out)
        return out

# 冻结函数
def freeze_lstm_layers(model, freeze_layers):
    for name, param in model.lstm.named_parameters():
        if any(f"l{layer}" in name for layer in freeze_layers):
            param.requires_grad = False
        else:
            param.requires_grad = True

# 组合
freeze_configs = {
    "0": [0],
    "01": [0, 1],
    "012": [0, 1, 2],
    "no": []
}

# 训练比例
train_ratios = [0.1 * i for i in range(1, 9)]
# train_ratios = [0.05 * i for i in range(1, 17)] 

# 加载数据
full_data = pd.read_csv(data_file).dropna()
total_len = len(full_data)
feature_scaler = joblib.load("feature_scaler_v3_b1.pkl")
target_scaler = joblib.load("target_scaler_v3_b1.pkl")

# 主循环
for freeze_tag, freeze_layers in freeze_configs.items():
    for ratio in train_ratios:
        ratio_str = f"{int(ratio*100)}%"
        experiment_dir = os.path.join(base_experiment_dir, f"exp_{freeze_tag}_{ratio_str}")
        os.makedirs(experiment_dir, exist_ok=True)
        print(f"Running experiment with freeze_layers: {freeze_layers}, ratio: {ratio_str}")

        # 划分数据
        train_data = full_data.iloc[:int(ratio * total_len)]
        val_data = full_data.iloc[int(0.8 * total_len):int(0.9 * total_len)]
        test_data = full_data.iloc[int(0.9 * total_len):]
        print(f"Train data length: {len(train_data)}, Validation data length: {len(val_data)}, Test data length: {len(test_data)}")

        # 特征和目标
        train_features = train_data.drop(columns=[timestamp_col])
        val_features = val_data.drop(columns=[timestamp_col])
        test_features = test_data.drop(columns=[timestamp_col])

        train_target = train_data[[target_feature]]
        val_target = val_data[[target_feature]]
        test_target = test_data[[target_feature]]

        train_features_scaled = feature_scaler.transform(train_features)
        val_features_scaled = feature_scaler.transform(val_features)
        test_features_scaled = feature_scaler.transform(test_features)
        train_target_scaled = target_scaler.transform(train_target)
        val_target_scaled = target_scaler.transform(val_target)
        test_target_scaled = target_scaler.transform(test_target)

        def preprocess_data(features, target):
            X, y = [], []
            for i in range(len(features) - n_past - n_future + 1):
                X.append(features[i:i + n_past])
                y.append(target[i + n_past:i + n_past + n_future])
            return np.array(X), np.array(y)

        train_X, train_y = preprocess_data(train_features_scaled, train_target_scaled)
        val_X, val_y = preprocess_data(val_features_scaled, val_target_scaled)
        test_X, test_y = preprocess_data(test_features_scaled, test_target_scaled)

        # 张量
        train_X_tensor = torch.tensor(train_X, dtype=torch.float32)
        train_y_tensor = torch.tensor(train_y, dtype=torch.float32).squeeze(-1)
        val_X_tensor = torch.tensor(val_X, dtype=torch.float32)
        val_y_tensor = torch.tensor(val_y, dtype=torch.float32).squeeze(-1)
        test_X_tensor = torch.tensor(test_X, dtype=torch.float32)
        test_y_tensor = torch.tensor(test_y, dtype=torch.float32).squeeze(-1)

        train_loader = DataLoader(TensorDataset(train_X_tensor, train_y_tensor), batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(TensorDataset(val_X_tensor, val_y_tensor), batch_size=batch_size, shuffle=False)

        # 模型
        input_dim = train_X_tensor.shape[2]
        output_dim = n_future
        model = LSTMModel(input_dim, hidden_dim, output_dim, num_layers, dropout).to(device)
        model.load_state_dict(torch.load("source_best_model_v3_b1.pth"))

        freeze_lstm_layers(model, freeze_layers)
        for param in model.fc.parameters():
            param.requires_grad = True

        optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate, weight_decay=1e-4)
        criterion = nn.MSELoss()
        train_losses, val_losses = [], []

        # Early stopping parameters
        patience = 10
        best_val_loss = float('inf')
        early_stop_counter = 0

        import time

        # Training loop with early stopping
        log_file = os.path.join(experiment_dir, f"training_log.txt")
        with open(log_file, "w") as log:
            for epoch in range(epochs):
                # print(time.time())
                model.train()
                train_loss_epoch = 0
                for batch_X, batch_y in train_loader:
                    batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                    optimizer.zero_grad()
                    output = model(batch_X)
                    loss = criterion(output, batch_y)
                    loss.backward()
                    optimizer.step()
                    train_loss_epoch += loss.item()
                train_loss_epoch /= len(train_loader)

                # Validation
                model.eval()
                val_loss_epoch = 0
                with torch.no_grad():
                    for batch_X, batch_y in val_loader:
                        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                        val_output = model(batch_X)
                        val_loss = criterion(val_output, batch_y)
                        val_loss_epoch += val_loss.item()
                val_loss_epoch /= len(val_loader)

                train_losses.append(train_loss_epoch)
                val_losses.append(val_loss_epoch)

                # Check for early stopping
                if val_loss_epoch < best_val_loss:
                    best_val_loss = val_loss_epoch
                    best_model_state = model.state_dict()
                    early_stop_counter = 0
                else:
                    early_stop_counter += 1
                    if early_stop_counter >= patience:
                        print(f"Early stopping triggered at epoch {epoch + 1}")
                        log.write(f"Early stopping triggered at epoch {epoch + 1}\n")
                        break

                log.write(f"Epoch {epoch + 1}/{epochs}, Train Loss: {train_loss_epoch:.4f}, Val Loss: {val_loss_epoch:.4f}\n")
                print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {train_loss_epoch:.4f}, Val Loss: {val_loss_epoch:.4f}")

        model.load_state_dict(best_model_state)
        torch.save(model.state_dict(), os.path.join(experiment_dir, f"transfer_best_model.pth"))

        # Save loss plot
        loss_plot_path = os.path.join(experiment_dir, "loss_plot.png")
        plt.figure()
        plt.plot(train_losses, label="Train Loss")
        plt.plot(val_losses, label="Val Loss")
        plt.legend()
        plt.title("Training and Validation Loss")
        plt.savefig(loss_plot_path)
        plt.close()


        # Direct multi-step forecasting function
        def direct_multi_step_forecast(model, test_X, target_scaler, device, use_last_timestep=False):
            model.eval()
            input_tensor = test_X.clone().detach().to(device)
            pred = model(input_tensor).cpu().detach().numpy()
            if use_last_timestep:
                pred_reshaped = pred[:, -1].reshape(-1, 1)
            else:
                pred_reshaped = pred.reshape(-1, 1)
            pred_original = target_scaler.inverse_transform(pred_reshaped)
            return pred_original.flatten()

        # Initialize data structures for combined CSV storage
        all_results = []
        all_predictions = []

        # Train, Validation, and Test predictions and plots
        for dataset_name, tensor_X, tensor_y in [("Train", train_X_tensor, train_y_tensor), 
                                                ("Validation", val_X_tensor, val_y_tensor), 
                                                ("Test", test_X_tensor, test_y_tensor)]:
            # Option 1: Predictions using pred.reshape(-1, 1)
            preds_option1 = direct_multi_step_forecast(model, tensor_X, target_scaler, device, use_last_timestep=False)
            true_values_option1 = target_scaler.inverse_transform(tensor_y.reshape(-1, 1)).flatten()

            # Option 2: Predictions using pred[:, -1].reshape(-1, 1)
            preds_option2 = direct_multi_step_forecast(model, tensor_X, target_scaler, device, use_last_timestep=True)
            true_values_option2 = target_scaler.inverse_transform(tensor_y[:, -1].numpy().reshape(-1, 1)).flatten()

            # Metrics for both options
            mse_option1 = mean_squared_error(true_values_option1, preds_option1)
            r2_option1 = r2_score(true_values_option1, preds_option1)
            mse_option2 = mean_squared_error(true_values_option2, preds_option2)
            r2_option2 = r2_score(true_values_option2, preds_option2)

            # Save metrics to a combined results list
            all_results.append({
                "Dataset": dataset_name,
                "Option": "Option 1",
                "MSE": mse_option1,
                "R2": r2_option1
            })
            all_results.append({
                "Dataset": dataset_name,
                "Option": "Option 2",
                "MSE": mse_option2,
                "R2": r2_option2
            })

            # Save predictions and true values for combined output
            for idx, (preds, true_values, option) in enumerate([
                (preds_option1, true_values_option1, "Option 1"),
                (preds_option2, true_values_option2, "Option 2"),
            ]):
                dataset_length = len(true_values)
                all_predictions.extend([
                    {
                        "Dataset": dataset_name,
                        "Option": option,
                        "Index": i,
                        "True Value": true_values[i],
                        "Predicted Value": preds[i]
                    }
                    for i in range(dataset_length)
                ])

            # Visualization for both options
            for option, preds, true_values, mse, r2 in [
                ("option1", preds_option1, true_values_option1, mse_option1, r2_option1),
                ("option2", preds_option2, true_values_option2, mse_option2, r2_option2)
            ]:
                # Line plot: True vs Predicted
                line_chart_filename = os.path.join(experiment_dir, f"{dataset_name.lower()}_line_chart_{option}.png")
                plt.figure()
                plt.plot(range(len(true_values)), true_values, label="True Values", color="green", linewidth=2)
                plt.plot(range(len(preds)), preds, label="Predicted Values", color="orange", linestyle="--", linewidth=2)
                plt.text(
                    0.05, 0.95, f"MSE: {mse:.4f}\nR2: {r2:.4f}",
                    transform=plt.gca().transAxes,
                    fontsize=10, verticalalignment='top',
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='black')
                )
                plt.title(f"{dataset_name} Predictions - {option} (Line Chart)")
                plt.xlabel("Index")
                plt.ylabel("Values")
                plt.legend()
                plt.savefig(line_chart_filename)
                plt.close()

                # Residuals histogram
                residuals = true_values - preds
                residuals_plot_filename = os.path.join(experiment_dir, f"{dataset_name.lower()}_residuals_{option}.png")
                plt.figure()
                plt.hist(residuals, bins=50, alpha=0.7, color="purple", label="Residuals")
                plt.title(f"{dataset_name} Residuals Distribution - {option}")
                plt.xlabel("Residuals")
                plt.ylabel("Frequency")
                plt.text(
                    0.05, 0.95, f"MSE: {mse:.4f}\nR2: {r2:.4f}",
                    transform=plt.gca().transAxes,
                    fontsize=10, verticalalignment='top',
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='black')
                )
                plt.legend()
                plt.savefig(residuals_plot_filename)
                plt.close()

            # Combined residuals histogram
            combined_residuals_plot_filename = os.path.join(experiment_dir, f"{dataset_name.lower()}_residuals_combined.png")
            plt.figure()
            plt.hist(true_values_option1 - preds_option1, bins=50, alpha=0.5, label="Option 1", color="blue")
            plt.hist(true_values_option2 - preds_option2, bins=50, alpha=0.5, label="Option 2", color="orange")
            plt.title(f"{dataset_name} Residuals Distribution - Combined")
            plt.xlabel("Residuals")
            plt.ylabel("Frequency")
            plt.legend()
            plt.savefig(combined_residuals_plot_filename)
            plt.close()

        # Collect all parameters and results
        experiment_parameters = {
            "input_dim": input_dim,
            "hidden_dim": hidden_dim,
            "output_dim": output_dim,
            "train_data": len(train_data),
            "val_data": len(val_data),
            "test_data": len(test_data),
            "freeze_layers": freeze_layers,
            "num_layers": num_layers,
            "dropout": dropout,
            "batch_size": batch_size,
            "learning_rate": learning_rate,
            "epochs": epochs,
            "timestamp_col": timestamp_col,
            "patience": patience,
            "n_past": n_past,
            "n_future": n_future,
            "optimizer": "Adam",
            "loss_function": "MSELoss",
            "device": device.type
        }

        # Combine results with parameters
        for result in all_results:
            result.update(experiment_parameters)

        # Save combined metrics and parameters to CSV
        results_csv_path = os.path.join(
            experiment_dir, f"metrics_results.csv")
        results_df = pd.DataFrame(all_results)
        results_df.to_csv(results_csv_path, index=False)

        # Save combined predictions to CSV
        predictions_csv_path = os.path.join(experiment_dir, f"predictions_results.csv")
        predictions_df = pd.DataFrame(all_predictions)
        predictions_df.to_csv(predictions_csv_path, index=False)

        print(f"Metrics saved to: {results_csv_path}")
        print(f"Predictions saved to: {predictions_csv_path}")
        print(f"All outputs saved to {experiment_dir}")
        print(f"Experiment {freeze_tag} with ratio {ratio_str} completed.\n")



# target

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch import nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from datetime import datetime
import joblib
import argparse
import time

parser = argparse.ArgumentParser()
parser.add_argument('--seed', type=int, required=True, help='Random seed')
args = parser.parse_args()

seed = args.seed
np.random.seed(seed)
torch.manual_seed(seed)

base_experiment_dir = f"experiments_target_v3_b2_{seed}"
os.makedirs(base_experiment_dir, exist_ok=True)

print(f"Using seed: {seed}")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# 参数设置
hidden_dim = 64
num_layers = 3
dropout = 0.2
epochs = 500
learning_rate = 5e-3
n_past = 30
n_future = 10
batch_size = 64
timestamp_col = 'timestamp'
target_feature = 'motor6_temperature'
data_file = '../data/target_all_data_b2.csv'

# 模型定义
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, dropout=dropout if num_layers > 1 else 0, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])
        out = self.fc(out)
        return out

# 训练比例
# train_ratios = [0.1 * i for i in range(1, 9)]
train_ratios = [0.05 * i for i in range(1, 17)] 

# 加载数据
full_data = pd.read_csv(data_file).dropna()
total_len = len(full_data)
# Scale features and target separately
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()




for ratio in train_ratios:
    ratio_str = f"{int(ratio*100)}%"
    experiment_dir = os.path.join(base_experiment_dir, f"exp_{ratio_str}")
    os.makedirs(experiment_dir, exist_ok=True)
    print(f"Running experiment with ratio: {ratio_str}")

    # 划分数据
    train_data = full_data.iloc[:int(ratio * total_len)]
    val_data = full_data.iloc[int(0.8 * total_len):int(0.9 * total_len)]
    test_data = full_data.iloc[int(0.9 * total_len):]
    print(f"Train data length: {len(train_data)}, Validation data length: {len(val_data)}, Test data length: {len(test_data)}")

    # 特征和目标
    train_features = train_data.drop(columns=[timestamp_col])
    val_features = val_data.drop(columns=[timestamp_col])
    test_features = test_data.drop(columns=[timestamp_col])

    train_target = train_data[[target_feature]]
    val_target = val_data[[target_feature]]
    test_target = test_data[[target_feature]]

    train_features_scaled = feature_scaler.fit_transform(train_features)
    val_features_scaled = feature_scaler.transform(val_features)
    test_features_scaled = feature_scaler.transform(test_features)
    train_target_scaled = target_scaler.fit_transform(train_target)
    val_target_scaled = target_scaler.transform(val_target)
    test_target_scaled = target_scaler.transform(test_target)

    def preprocess_data(features, target):
        X, y = [], []
        for i in range(len(features) - n_past - n_future + 1):
            X.append(features[i:i + n_past])
            y.append(target[i + n_past:i + n_past + n_future])
        return np.array(X), np.array(y)

    train_X, train_y = preprocess_data(train_features_scaled, train_target_scaled)
    val_X, val_y = preprocess_data(val_features_scaled, val_target_scaled)
    test_X, test_y = preprocess_data(test_features_scaled, test_target_scaled)

    # 张量
    train_X_tensor = torch.tensor(train_X, dtype=torch.float32)
    train_y_tensor = torch.tensor(train_y, dtype=torch.float32).squeeze(-1)
    val_X_tensor = torch.tensor(val_X, dtype=torch.float32)
    val_y_tensor = torch.tensor(val_y, dtype=torch.float32).squeeze(-1)
    test_X_tensor = torch.tensor(test_X, dtype=torch.float32)
    test_y_tensor = torch.tensor(test_y, dtype=torch.float32).squeeze(-1)

    train_loader = DataLoader(TensorDataset(train_X_tensor, train_y_tensor), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(val_X_tensor, val_y_tensor), batch_size=batch_size, shuffle=False)

    # 模型
    input_dim = train_X_tensor.shape[2]
    output_dim = n_future
    model = LSTMModel(input_dim, hidden_dim, output_dim, num_layers, dropout).to(device)


    # Training setup
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    train_losses, val_losses = [], []

    # Early stopping parameters
    patience = 10
    best_val_loss = float('inf')
    early_stop_counter = 0

    # Training loop with early stopping
    log_file = os.path.join(experiment_dir, f"training_log.txt")
    with open(log_file, "w") as log:
        for epoch in range(epochs):
            # print(time.time())
            model.train()
            train_loss_epoch = 0
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                optimizer.zero_grad()
                output = model(batch_X)
                loss = criterion(output, batch_y)
                loss.backward()
                optimizer.step()
                train_loss_epoch += loss.item()
            train_loss_epoch /= len(train_loader)

            # Validation
            model.eval()
            val_loss_epoch = 0
            with torch.no_grad():
                for batch_X, batch_y in val_loader:
                    batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                    val_output = model(batch_X)
                    val_loss = criterion(val_output, batch_y)
                    val_loss_epoch += val_loss.item()
            val_loss_epoch /= len(val_loader)

            train_losses.append(train_loss_epoch)
            val_losses.append(val_loss_epoch)

            # Check for early stopping
            if val_loss_epoch < best_val_loss:
                best_val_loss = val_loss_epoch
                best_model_state = model.state_dict()
                early_stop_counter = 0
            else:
                early_stop_counter += 1
                if early_stop_counter >= patience:
                    print(f"Early stopping triggered at epoch {epoch + 1}")
                    log.write(f"Early stopping triggered at epoch {epoch + 1}\n")
                    break

            log.write(f"Epoch {epoch + 1}/{epochs}, Train Loss: {train_loss_epoch:.4f}, Val Loss: {val_loss_epoch:.4f}\n")
            print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {train_loss_epoch:.4f}, Val Loss: {val_loss_epoch:.4f}")

    model.load_state_dict(best_model_state)
    torch.save(model.state_dict(), os.path.join(experiment_dir, f"target_best_model.pth"))

    # Save loss plot
    loss_plot_path = os.path.join(experiment_dir, "loss_plot.png")
    plt.figure()
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Val Loss")
    plt.legend()
    plt.title("Training and Validation Loss")
    plt.savefig(loss_plot_path)
    plt.close()


    # Direct multi-step forecasting function
    def direct_multi_step_forecast(model, test_X, target_scaler, device, use_last_timestep=False):
        model.eval()
        input_tensor = test_X.clone().detach().to(device)
        pred = model(input_tensor).cpu().detach().numpy()
        if use_last_timestep:
            pred_reshaped = pred[:, -1].reshape(-1, 1)
        else:
            pred_reshaped = pred.reshape(-1, 1)
        pred_original = target_scaler.inverse_transform(pred_reshaped)
        return pred_original.flatten()

    # Initialize data structures for combined CSV storage
    all_results = []
    all_predictions = []

    # Train, Validation, and Test predictions and plots
    for dataset_name, tensor_X, tensor_y in [("Train", train_X_tensor, train_y_tensor), 
                                            ("Validation", val_X_tensor, val_y_tensor), 
                                            ("Test", test_X_tensor, test_y_tensor)]:
        # Option 1: Predictions using pred.reshape(-1, 1)
        preds_option1 = direct_multi_step_forecast(model, tensor_X, target_scaler, device, use_last_timestep=False)
        true_values_option1 = target_scaler.inverse_transform(tensor_y.reshape(-1, 1)).flatten()

        # Option 2: Predictions using pred[:, -1].reshape(-1, 1)
        preds_option2 = direct_multi_step_forecast(model, tensor_X, target_scaler, device, use_last_timestep=True)
        true_values_option2 = target_scaler.inverse_transform(tensor_y[:, -1].numpy().reshape(-1, 1)).flatten()

        # Metrics for both options
        mse_option1 = mean_squared_error(true_values_option1, preds_option1)
        r2_option1 = r2_score(true_values_option1, preds_option1)
        mse_option2 = mean_squared_error(true_values_option2, preds_option2)
        r2_option2 = r2_score(true_values_option2, preds_option2)

        # Save metrics to a combined results list
        all_results.append({
            "Dataset": dataset_name,
            "Option": "Option 1",
            "MSE": mse_option1,
            "R2": r2_option1
        })
        all_results.append({
            "Dataset": dataset_name,
            "Option": "Option 2",
            "MSE": mse_option2,
            "R2": r2_option2
        })

        # Save predictions and true values for combined output
        for idx, (preds, true_values, option) in enumerate([
            (preds_option1, true_values_option1, "Option 1"),
            (preds_option2, true_values_option2, "Option 2"),
        ]):
            dataset_length = len(true_values)
            all_predictions.extend([
                {
                    "Dataset": dataset_name,
                    "Option": option,
                    "Index": i,
                    "True Value": true_values[i],
                    "Predicted Value": preds[i]
                }
                for i in range(dataset_length)
            ])

        # Visualization for both options
        for option, preds, true_values, mse, r2 in [
            ("option1", preds_option1, true_values_option1, mse_option1, r2_option1),
            ("option2", preds_option2, true_values_option2, mse_option2, r2_option2)
        ]:
            # Line plot: True vs Predicted
            line_chart_filename = os.path.join(experiment_dir, f"{dataset_name.lower()}_line_chart_{option}.png")
            plt.figure()
            plt.plot(range(len(true_values)), true_values, label="True Values", color="green", linewidth=2)
            plt.plot(range(len(preds)), preds, label="Predicted Values", color="orange", linestyle="--", linewidth=2)
            plt.text(
                0.05, 0.95, f"MSE: {mse:.4f}\nR2: {r2:.4f}",
                transform=plt.gca().transAxes,
                fontsize=10, verticalalignment='top',
                bbox=dict(facecolor='white', alpha=0.8, edgecolor='black')
            )
            plt.title(f"{dataset_name} Predictions - {option} (Line Chart)")
            plt.xlabel("Index")
            plt.ylabel("Values")
            plt.legend()
            plt.savefig(line_chart_filename)
            plt.close()

            # Residuals histogram
            residuals = true_values - preds
            residuals_plot_filename = os.path.join(experiment_dir, f"{dataset_name.lower()}_residuals_{option}.png")
            plt.figure()
            plt.hist(residuals, bins=50, alpha=0.7, color="purple", label="Residuals")
            plt.title(f"{dataset_name} Residuals Distribution - {option}")
            plt.xlabel("Residuals")
            plt.ylabel("Frequency")
            plt.text(
                0.05, 0.95, f"MSE: {mse:.4f}\nR2: {r2:.4f}",
                transform=plt.gca().transAxes,
                fontsize=10, verticalalignment='top',
                bbox=dict(facecolor='white', alpha=0.8, edgecolor='black')
            )
            plt.legend()
            plt.savefig(residuals_plot_filename)
            plt.close()

        # Combined residuals histogram
        combined_residuals_plot_filename = os.path.join(experiment_dir, f"{dataset_name.lower()}_residuals_combined.png")
        plt.figure()
        plt.hist(true_values_option1 - preds_option1, bins=50, alpha=0.5, label="Option 1", color="blue")
        plt.hist(true_values_option2 - preds_option2, bins=50, alpha=0.5, label="Option 2", color="orange")
        plt.title(f"{dataset_name} Residuals Distribution - Combined")
        plt.xlabel("Residuals")
        plt.ylabel("Frequency")
        plt.legend()
        plt.savefig(combined_residuals_plot_filename)
        plt.close()

    # Collect all parameters and results
    experiment_parameters = {
        "input_dim": input_dim,
        "hidden_dim": hidden_dim,
        "output_dim": output_dim,
        "train_data": len(train_data),
        "val_data": len(val_data),
        "test_data": len(test_data),
        "num_layers": num_layers,
        "dropout": dropout,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "epochs": early_stop_counter,
        "timestamp_col": timestamp_col,
        "patience": patience,
        "n_past": n_past,
        "n_future": n_future,
        "optimizer": "Adam",
        "loss_function": "MSELoss",
        "device": device.type
    }

    # Combine results with parameters
    for result in all_results:
        result.update(experiment_parameters)

    # Save combined metrics and parameters to CSV
    results_csv_path = os.path.join(
        experiment_dir, f"metrics_results.csv")
    results_df = pd.DataFrame(all_results)
    results_df.to_csv(results_csv_path, index=False)

    # Save combined predictions to CSV
    predictions_csv_path = os.path.join(experiment_dir, f"predictions_results.csv")
    predictions_df = pd.DataFrame(all_predictions)
    predictions_df.to_csv(predictions_csv_path, index=False)

    print(f"Metrics saved to: {results_csv_path}")
    print(f"Predictions saved to: {predictions_csv_path}")
    print(f"All outputs saved to {experiment_dir}")
    print(f"Experiment ratio {ratio_str} completed.\n")



# experiments_direct_generalization_v3_b1

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch import nn
from sklearn.metrics import mean_squared_error, r2_score
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import joblib

# ============================================================
# Basic setup (Direct generalization: one-shot inference only)
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Output directory (one-shot evaluation)
base_experiment_dir = "experiments_direct_generalization_v3_b1"
os.makedirs(base_experiment_dir, exist_ok=True)

# ============================================================
# Config
# ============================================================
hidden_dim = 64
num_layers = 3
dropout = 0.2

n_past = 30
n_future = 10
batch_size = 256  # inference can be larger if GPU memory allows

timestamp_col = "timestamp"
target_feature = "motor6_temperature"

data_file = "../data/target_all_data_b1.csv"
feature_scaler_path = "feature_scaler_v3_b1.pkl"
target_scaler_path = "target_scaler_v3_b1.pkl"
source_ckpt_path = "source_best_model_v3_b1.pth"

# ============================================================
# Model
# ============================================================
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])  # last timestep
        out = self.fc(out)                 # [B, n_future]
        return out

# ============================================================
# Utilities
# ============================================================
def preprocess_data(features_np: np.ndarray, target_np: np.ndarray, n_past: int, n_future: int):
    """
    features_np: shape [T, F]
    target_np:   shape [T, 1] or [T]
    Returns:
      X: [N, n_past, F]
      y: [N, n_future, 1]
    """
    if target_np.ndim == 1:
        target_np = target_np.reshape(-1, 1)

    X, y = [], []
    T = len(features_np)
    last_start = T - n_past - n_future + 1
    for i in range(last_start):
        X.append(features_np[i:i + n_past])
        y.append(target_np[i + n_past:i + n_past + n_future])
    return np.array(X), np.array(y)

@torch.no_grad()
def predict_direct(model: nn.Module, X_tensor: torch.Tensor, device: torch.device, batch_size: int):
    """
    Returns raw scaled predictions:
      pred: numpy array [N, n_future]
    """
    model.eval()
    loader = DataLoader(TensorDataset(X_tensor), batch_size=batch_size, shuffle=False)
    preds = []
    for (batch_X,) in loader:
        batch_X = batch_X.to(device)
        out = model(batch_X)  # [B, n_future]
        preds.append(out.detach().cpu().numpy())
    return np.concatenate(preds, axis=0)

def inverse_transform_flat(target_scaler, arr_2d):
    """
    arr_2d: shape [K, 1]
    returns: shape [K]
    """
    return target_scaler.inverse_transform(arr_2d).reshape(-1)

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1e-8):
    """
    y_true, y_pred: 1D numpy arrays in original scale
    Returns: MSE, RMSE, R2, MAPE(%)
    """
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    r2 = r2_score(y_true, y_pred)
    mape = float(np.mean(np.abs((y_true - y_pred) / (y_true + eps))) * 100.0)
    return float(mse), rmse, float(r2), mape

def save_line_plot(save_path, true_values, pred_values, title, mse, rmse, r2, mape):
    plt.figure()
    plt.plot(range(len(true_values)), true_values, label="True Values", linewidth=2)
    plt.plot(range(len(pred_values)), pred_values, label="Predicted Values", linestyle="--", linewidth=2)
    plt.title(title)
    plt.xlabel("Index")
    plt.ylabel("Value")
    plt.legend()
    plt.text(
        0.05, 0.95, f"MSE: {mse:.6f}\nRMSE: {rmse:.6f}\nR2: {r2:.6f}\nMAPE: {mape:.2f}%",
        transform=plt.gca().transAxes,
        fontsize=10, verticalalignment="top",
        bbox=dict(facecolor="white", alpha=0.8, edgecolor="black"),
    )
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()

def save_residual_hist(save_path, residuals, title, mse, rmse, r2, mape):
    plt.figure()
    plt.hist(residuals, bins=50, alpha=0.7, label="Residuals")
    plt.title(title)
    plt.xlabel("Residual (True - Pred)")
    plt.ylabel("Frequency")
    plt.legend()
    plt.text(
        0.05, 0.95, f"MSE: {mse:.6f}\nRMSE: {rmse:.6f}\nR2: {r2:.6f}\nMAPE: {mape:.2f}%",
        transform=plt.gca().transAxes,
        fontsize=10, verticalalignment="top",
        bbox=dict(facecolor="white", alpha=0.8, edgecolor="black"),
    )
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()

# ============================================================
# 1) Load data & scalers
# ============================================================
full_data = pd.read_csv(data_file).dropna().reset_index(drop=True)
total_len = len(full_data)
print("Total rows:", total_len)

feature_scaler = joblib.load(feature_scaler_path)
target_scaler = joblib.load(target_scaler_path)

# ============================================================
# 2) Fixed split (same as your original: last 10% is test)
# ============================================================
test_data = full_data.iloc[int(0.9 * total_len):].copy()
print("Test rows:", len(test_data))

test_features = test_data.drop(columns=[timestamp_col])
test_target = test_data[[target_feature]]

test_features_scaled = feature_scaler.transform(test_features)
test_target_scaled = target_scaler.transform(test_target)

# Windowing
test_X, test_y = preprocess_data(test_features_scaled, test_target_scaled, n_past=n_past, n_future=n_future)

test_X_tensor = torch.tensor(test_X, dtype=torch.float32)
test_y_tensor = torch.tensor(test_y, dtype=torch.float32).squeeze(-1)  # [N, n_future]

print("Test windows X:", test_X_tensor.shape, "y:", test_y_tensor.shape)

# ============================================================
# 3) Load source model once
# ============================================================
input_dim = test_X_tensor.shape[2]
output_dim = n_future
model = LSTMModel(input_dim, hidden_dim, output_dim, num_layers, dropout).to(device)

state = torch.load(source_ckpt_path, map_location=device)
model.load_state_dict(state)
model.eval()
print("Loaded source checkpoint:", source_ckpt_path)

# ============================================================
# 4) Inference on test set
# ============================================================
pred_scaled = predict_direct(model, test_X_tensor, device=device, batch_size=batch_size)  # [N, n_future]

# Option 1: flatten all horizons (N*n_future points)
preds_option1 = inverse_transform_flat(target_scaler, pred_scaled.reshape(-1, 1))
true_option1 = inverse_transform_flat(target_scaler, test_y_tensor.reshape(-1, 1).numpy())

mse_o1, rmse_o1, r2_o1, mape_o1 = compute_metrics(true_option1, preds_option1)

# Option 2: only last step of each horizon (N points)
pred_last_scaled = pred_scaled[:, -1].reshape(-1, 1)
true_last_scaled = test_y_tensor[:, -1].numpy().reshape(-1, 1)

preds_option2 = inverse_transform_flat(target_scaler, pred_last_scaled)
true_option2 = inverse_transform_flat(target_scaler, true_last_scaled)

mse_o2, rmse_o2, r2_o2, mape_o2 = compute_metrics(true_option2, preds_option2)

print(f"[Test | Option1] MSE={mse_o1:.6f}, RMSE={rmse_o1:.6f}, R2={r2_o1:.6f}, MAPE={mape_o1:.2f}%")
print(f"[Test | Option2] MSE={mse_o2:.6f}, RMSE={rmse_o2:.6f}, R2={r2_o2:.6f}, MAPE={mape_o2:.2f}%")

# ============================================================
# 5) Save outputs
# ============================================================
experiment_dir = os.path.join(base_experiment_dir, "exp_source_direct_test")
os.makedirs(experiment_dir, exist_ok=True)

# 5.1 Metrics CSV (4 metrics)
metrics = [
    {
        "Dataset": "Test",
        "Option": "Option 1 (flatten all steps)",
        "MSE": mse_o1,
        "RMSE": rmse_o1,
        "R2": r2_o1,
        "MAPE(%)": mape_o1,
        "n_past": n_past,
        "n_future": n_future,
        "test_split": "last 10%",
        "checkpoint": os.path.basename(source_ckpt_path),
        "device": device.type,
        "batch_size": batch_size,
    },
    {
        "Dataset": "Test",
        "Option": "Option 2 (last step only)",
        "MSE": mse_o2,
        "RMSE": rmse_o2,
        "R2": r2_o2,
        "MAPE(%)": mape_o2,
        "n_past": n_past,
        "n_future": n_future,
        "test_split": "last 10%",
        "checkpoint": os.path.basename(source_ckpt_path),
        "device": device.type,
        "batch_size": batch_size,
    },
]
metrics_df = pd.DataFrame(metrics)
metrics_csv_path = os.path.join(experiment_dir, "metrics_results.csv")
metrics_df.to_csv(metrics_csv_path, index=False)

# 5.2 Predictions CSV (store both options)
pred_rows = []

# Option 1 rows
for i in range(len(true_option1)):
    pred_rows.append({
        "Dataset": "Test",
        "Option": "Option 1 (flatten all steps)",
        "Index": i,
        "True Value": float(true_option1[i]),
        "Predicted Value": float(preds_option1[i]),
    })

# Option 2 rows
for i in range(len(true_option2)):
    pred_rows.append({
        "Dataset": "Test",
        "Option": "Option 2 (last step only)",
        "Index": i,
        "True Value": float(true_option2[i]),
        "Predicted Value": float(preds_option2[i]),
    })

pred_df = pd.DataFrame(pred_rows)
pred_csv_path = os.path.join(experiment_dir, "predictions_results.csv")
pred_df.to_csv(pred_csv_path, index=False)

# 5.3 Plots (4 plots)
save_line_plot(
    save_path=os.path.join(experiment_dir, "test_line_chart_option1.png"),
    true_values=true_option1,
    pred_values=preds_option1,
    title="Test Predictions - Option 1 (Flatten all steps)",
    mse=mse_o1, rmse=rmse_o1, r2=r2_o1, mape=mape_o1
)
save_residual_hist(
    save_path=os.path.join(experiment_dir, "test_residuals_option1.png"),
    residuals=true_option1 - preds_option1,
    title="Test Residuals - Option 1 (Flatten all steps)",
    mse=mse_o1, rmse=rmse_o1, r2=r2_o1, mape=mape_o1
)

save_line_plot(
    save_path=os.path.join(experiment_dir, "test_line_chart_option2.png"),
    true_values=true_option2,
    pred_values=preds_option2,
    title="Test Predictions - Option 2 (Last step only)",
    mse=mse_o2, rmse=rmse_o2, r2=r2_o2, mape=mape_o2
)
save_residual_hist(
    save_path=os.path.join(experiment_dir, "test_residuals_option2.png"),
    residuals=true_option2 - preds_option2,
    title="Test Residuals - Option 2 (Last step only)",
    mse=mse_o2, rmse=rmse_o2, r2=r2_o2, mape=mape_o2
)

print("Saved metrics to:", metrics_csv_path)
print("Saved predictions to:", pred_csv_path)
print("Saved plots to:", experiment_dir)
print("Direct generalization evaluation completed.")


# 源域验证集和测试集在2.2情况下的验证

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch import nn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib

# ============================================================
# Config (EDIT THESE)
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Source fault-injected datasets
src_val_fault_file = "../data/source_val_faults20_T_Th1_FL30-50_TI4-8_b1.csv"
src_test_fault_file = "../data/source_test_faults20_T_Th1_FL30-50_TI4-8_b1.csv"

# Scalers used during training (IMPORTANT: must match the model training)
feature_scaler_path = "feature_scaler_v3_b1.pkl"
target_scaler_path  = "target_scaler_v3_b1.pkl"

# Source model checkpoint (trained on healthy source data)
source_ckpt_path = "source_best_model_v3_b1.pth"

# Columns
timestamp_col = "timestamp"
label_col = "label"                 # your fault label column (0/1)
target_feature = "motor6_temperature"

# Window params
n_past = 30
n_future = 10
hidden_dim = 64
num_layers = 3
batch_size = 64  # not used much here because we do sequential rolling

# Fault decision threshold
tau_res = 2.2

# PROF-like input correction toggle (your behavior)
enable_feedback = True

# Output dir
output_dir = "results/source_fault_detection_tau2p2"
os.makedirs(output_dir, exist_ok=True)

# ============================================================
# Utilities
# ============================================================
def preprocess_data(features: np.ndarray, target: np.ndarray, n_past: int, n_future: int):
    """
    features: [T, F]
    target:   [T, 1] (scaled)
    Returns:
      X: [N, n_past, F]
      y: [N, n_future, 1]
    """
    X, y = [], []
    for i in range(len(features) - n_past - n_future + 1):
        X.append(features[i:i + n_past])
        y.append(target[i + n_past:i + n_past + n_future])
    return np.array(X), np.array(y)

def compute_metrics(y_true_bin: np.ndarray, y_pred_bin: np.ndarray):
    acc = float(accuracy_score(y_true_bin, y_pred_bin))
    prec = float(precision_score(y_true_bin, y_pred_bin, zero_division=0))
    rec = float(recall_score(y_true_bin, y_pred_bin, zero_division=0))
    f1 = float(f1_score(y_true_bin, y_pred_bin, zero_division=0))

    TP = int(np.sum((y_true_bin == 1) & (y_pred_bin == 1)))
    FP = int(np.sum((y_true_bin == 0) & (y_pred_bin == 1)))
    FN = int(np.sum((y_true_bin == 1) & (y_pred_bin == 0)))
    TN = int(np.sum((y_true_bin == 0) & (y_pred_bin == 0)))
    return acc, prec, rec, f1, TN, FP, FN, TP

# ============================================================
# Model (match your style, but FIX forward)
# ============================================================
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        """
        x: [B, T, F]
        return: [B, n_future]
        """
        out, _ = self.lstm(x)         # out: [B, T, H]
        out_last = out[:, -1, :]      # [B, H]  <-- correct
        out_fc = self.fc(out_last)    # [B, n_future]
        return out_fc

@torch.no_grad()
def predict_one_window(model: nn.Module, window_feats_scaled: torch.Tensor, device):
    """
    window_feats_scaled: [n_past, F] (torch tensor, scaled)
    Returns:
      pred_scaled: [n_future] numpy (scaled target space)
    """
    model.eval()
    x = window_feats_scaled.unsqueeze(0).to(device)  # [1, n_past, F]
    pred = model(x).detach().cpu().numpy().reshape(-1)  # [n_future]
    return pred

# ============================================================
# Core evaluation (match your rolling logic + last-step residual)
# ============================================================
def eval_fault_detection_on_csv(dataset_name: str, csv_path: str, model: nn.Module,
                                feature_scaler, target_scaler):
    df = pd.read_csv(csv_path).dropna().reset_index(drop=True)

    # sanity checks
    for col in [timestamp_col, label_col, target_feature]:
        if col not in df.columns:
            raise ValueError(f"[{dataset_name}] Missing required column '{col}' in {csv_path}. "
                             f"Available columns: {list(df.columns)}")

    # IMPORTANT:
    # Match your pipeline: features drop timestamp + label ONLY (keep motor6_temperature inside inputs)
    feature_df = df.drop(columns=[timestamp_col, label_col]).copy()
    target_df  = df[[target_feature]].copy()
    labels = df[label_col].astype(int).to_numpy()  # 0/1 per timestamp

    # Scale using pre-fitted scalers
    feat_scaled = feature_scaler.transform(feature_df)            # [T, F]
    tgt_scaled  = target_scaler.transform(target_df)              # [T, 1]

    # Determine the column index of motor6_temperature inside features (for feedback update)
    feat_cols = list(feature_df.columns)
    if target_feature not in feat_cols:
        raise ValueError(f"[{dataset_name}] target_feature '{target_feature}' is not in feature columns. "
                         f"Your current pipeline assumes it IS included in inputs (for feedback). "
                         f"Feature columns: {feat_cols[:10]} ... (total {len(feat_cols)})")
    temp_feat_idx = feat_cols.index(target_feature)

    # Build windows for y (only to get true last step cleanly)
    # We do not directly use X from preprocess_data because we apply rolling feedback.
    # But we still need y for the true last step in original scale.
    _, y_scaled = preprocess_data(feat_scaled, tgt_scaled, n_past, n_future)  # y_scaled: [N, n_future, 1]
    N = y_scaled.shape[0]

    # Labels synchronized to "last step of horizon"
    # horizon last step corresponds to timestamp i + n_past + n_future - 1
    labels_sync = labels[n_past + n_future - 1:]  # length should be N
    assert len(labels_sync) == N, f"[{dataset_name}] label sync length mismatch: {len(labels_sync)} vs {N}"

    # Rolling evaluation with optional feedback update
    feat_scaled_roll = torch.tensor(feat_scaled, dtype=torch.float32)  # [T, F]
    y_scaled_torch = torch.tensor(y_scaled, dtype=torch.float32)       # [N, n_future, 1]

    pred_last_list = []
    true_last_list = []
    residual_list  = []
    pred_label_list = []

    for i in range(N):
        # window input: [n_past, F]
        window = feat_scaled_roll[i:i + n_past]  # torch tensor
        pred_scaled = predict_one_window(model, window, device=device)  # [n_future] scaled

        # true last (scaled -> original)
        true_last_scaled = y_scaled_torch[i, -1, 0].item()
        pred_last_scaled = float(pred_scaled[-1])

        true_last = target_scaler.inverse_transform(np.array([[true_last_scaled]])).reshape(-1)[0]
        pred_last = target_scaler.inverse_transform(np.array([[pred_last_scaled]])).reshape(-1)[0]

        residual = abs(true_last - pred_last)
        pred_label = 1 if residual > tau_res else 0

        true_last_list.append(true_last)
        pred_last_list.append(pred_last)
        residual_list.append(residual)
        pred_label_list.append(pred_label)

        # Feedback update (your logic):
        # update the input at time idx = i + n_past + n_future - 1 (the last step aligned with label)
        # set motor6_temperature feature (scaled) to predicted last-step (scaled)
        if enable_feedback and pred_label == 1:
            update_t = i + n_past + n_future - 1
            # write scaled prediction back into feature matrix
            feat_scaled_roll[update_t, temp_feat_idx] = torch.tensor(pred_last_scaled, dtype=torch.float32)

    y_true_bin = labels_sync.astype(int)
    y_pred_bin = np.array(pred_label_list, dtype=int)

    acc, prec, rec, f1, TN, FP, FN, TP = compute_metrics(y_true_bin, y_pred_bin)

    # Save per-sample results
    out_df = pd.DataFrame({
        "Index": np.arange(N),
        "TrueTemp_Last": true_last_list,
        "PredTemp_Last": pred_last_list,
        "ResidualAbs_Last": residual_list,
        "TrueLabel": y_true_bin,
        "PredLabel": y_pred_bin
    })
    out_csv = os.path.join(output_dir, f"{dataset_name}_per_sample.csv")
    out_df.to_csv(out_csv, index=False)

    # Return summary row
    row = {
        "Dataset": dataset_name,
        "CSV": os.path.basename(csv_path),
        "tau_res": tau_res,
        "enable_feedback": int(enable_feedback),
        "n_past": n_past,
        "n_future": n_future,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "TN": TN,
        "FP": FP,
        "FN": FN,
        "TP": TP,
        "TotalSamples": int(N),
        "ActualFaults": int(y_true_bin.sum())
    }

    print(f"[{dataset_name}] Acc={acc:.6f}, Prec={prec:.6f}, Rec={rec:.6f}, F1={f1:.6f} | "
          f"TN={TN}, FP={FP}, FN={FN}, TP={TP} | N={N}")
    print(f"Saved per-sample CSV: {out_csv}")
    return row

# ============================================================
# Main
# ============================================================
if __name__ == "__main__":
    print("Loading scalers...")
    feature_scaler = joblib.load(feature_scaler_path)
    target_scaler  = joblib.load(target_scaler_path)

    # Infer input_dim from scaler (most robust)
    # If scaler was fit on numpy, it still has n_features_in_
    input_dim = int(getattr(feature_scaler, "n_features_in_", None))
    if input_dim is None:
        raise ValueError("Cannot infer input_dim from feature_scaler. Please check your scaler file.")
    print("Scaler expects input_dim =", input_dim)

    # Build model
    model = LSTMModel(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=n_future, num_layers=num_layers).to(device)

    # Load checkpoint
    state = torch.load(source_ckpt_path, map_location=device)
    model.load_state_dict(state)
    model.eval()
    print("Loaded source checkpoint:", source_ckpt_path)

    # Evaluate on source val-fault and test-fault
    summary_rows = []
    summary_rows.append(eval_fault_detection_on_csv("D_src_val_fault",  src_val_fault_file,  model, feature_scaler, target_scaler))
    summary_rows.append(eval_fault_detection_on_csv("D_src_test_fault", src_test_fault_file, model, feature_scaler, target_scaler))

    # Save summary
    summary_df = pd.DataFrame(summary_rows)
    summary_csv = os.path.join(output_dir, "metrics_summary.csv")
    summary_df.to_csv(summary_csv, index=False)
    print("Saved summary metrics:", summary_csv)

# 消融实验